# FreeFine Final Exhaustive Run — Account A (Structural / Spectral) — v2

**Goal:** finish the structural side of the 2D geometric-editing search. Every generated model is run on the **same historical balanced-200** (67 move / 66 rotate / 67 resize), then scored task-by-task.

This notebook intentionally includes conservative and high-risk arms:
- R1 reference
- HF30 and FULL30
- MID and MID+HF bands
- fixed-reference timing
- directional high-frequency weighting

All arms are resume-safe under Kaggle's runtime limit. Use **T4×2**. If a deadline stops remaining jobs, run the notebook again: completed images are detected and skipped.

## 1. Clone + timer

In [1]:
# ===== C1 clone + clock =====
import time, subprocess
NB_START=time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git && cd FreeFine && git checkout -q 4c9fdb971572b32edbeac13464659274c28decbb", shell=True, check=True); print("cloned + pinned FreeFine 4c9fdb971572")

cloned + pinned FreeFine 4c9fdb971572


## 2. Generation environment — identical versions to prior combined-model run

In [2]:
%%bash
# ===== C2 freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 83.8 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.39s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 845ms
 Downloaded torchvision
 Downloaded triton
 Downloaded pillow
 Downloaded networkx
 Downloaded numpy
 Downloaded sympy
 Downloaded torch
Prepared 18 packages in 36.14s
Installed 18 packages in 270ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

## 3. Metric environment — identical versions to prior combined-model run

In [3]:
%%bash
# ===== C3 metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK"

metric_env OK


Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 778ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-nvjitlink-cu12
 Downloaded pillow
 Downloaded networkx
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-curand-cu12
 Downloaded triton
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded numpy
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded torch
Prepared 27 packages in 44.59s
Installed 27 packages in 308ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4

## 4. Reproducibility/metric patches — identical seeded-MD setup

In [4]:
# ===== C4 patch metrics (args.3d, SD-2.1 mirror, SEEDED MD) + model.py start_layer =====
import pathlib, re
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [mr/"MD"/"mean_distance.py", mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=mr/"MD"/"mean_distance.py"; s=md.read_text()
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st, os as _os; _seed=int(_os.environ.get('FF_MD_SEED','42')); _st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)",1); md.write_text(s)
mm=pathlib.Path("/kaggle/temp/FreeFine/src/demo/model.py"); g=mm.read_text()
if not re.search(r'^\s*import os\b', g, re.M): g="import os\n"+g
assert "list(range(10, 16))" in g, "layer_idx hardcode missing"
n=g.count("list(range(10, 16))")
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
mm.write_text(g); print(f"patched metrics + model.py start_layer ({n} sites)")

patched metrics + model.py start_layer (5 sites)


## 5. Parameterize the released 2D inference script

In [5]:
# ===== C5 parametrize inference script =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"','pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src, "ori_mask/obj_label block mismatch"; src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src)
assert all(x in src for x in ["FF_USE_PROMPT","FF_GUIDANCE","FF_START_STEP"]), "parametrize failed"
print("parametrized script written")

parametrized script written


## 6. Final Account A patch — generalized spectral feature fusion

**v3 hotfix:** FreeFine is pinned to commit `4c9fdb971572b32edbeac13464659274c28decbb`; the `forward_sampling` UNet call is located with a whitespace-tolerant regex instead of a brittle two-line literal anchor.


In [6]:

# ===== FINAL A PATCH v2: R1 + generalized masked spectral feature fusion =====
import os, re, py_compile

S="/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"
M="/kaggle/temp/FreeFine/src/demo/model.py"
A="/kaggle/temp/FreeFine/src/utils/attention.py"

s=open(S).read()
m=open(M).read()
a=open(A).read()

def replace_once(text, old, new, name):
    assert old in text, f"ANCHOR NOT FOUND: {name}"
    assert new not in text, f"ALREADY PATCHED: {name}"
    return text.replace(old,new,1)

router_old = "        edit_param = case['edit_param']"
router_new = '''        edit_param = case['edit_param']
        _dx,_dy,_dz,_rx,_ry,_rz,_sx,_sy,_sz = edit_param
        if abs(float(_rz))>1e-6:
            _etype='rotate'
        elif abs(float(_sx)-1)>1e-6 or abs(float(_sy)-1)>1e-6:
            _etype='resize'
        else:
            _etype='move'
        os.environ['FF_CASE_TYPE']=_etype
        os.environ['FF_CASE_RZ']=str(float(_rz))

        for _k in ('FF_PRESERVE','FF_PRESERVE_W','FF_USE_PROMPT','FF_GUIDANCE'):
            os.environ.pop(_k,None)

        if os.environ.get('FF_ROUTER','0')=='1':
            if _etype=='move':
                os.environ['FF_PRESERVE']='1'
                os.environ['FF_PRESERVE_W']='0.3'
            elif _etype=='resize':
                os.environ['FF_USE_PROMPT']='1'
                os.environ['FF_GUIDANCE']='10'

        _ff_types={x.strip() for x in os.environ.get('FF_HFF_TYPES','').split(',') if x.strip()}
        os.environ['FF_HFF_ACTIVE'] = (
            '1' if os.environ.get('FF_HFF','0')=='1' and _etype in _ff_types else '0'
        )'''
s=replace_once(s,router_old,router_new,"R1 + per-case spectral switch")

pres_old = (
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                latents_list.append(latents)"
)
pres_new = (
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                if os.environ.get('FF_PRESERVE')=='1' and latents.shape[0]==2:\n"
"                    _cl = refer_latents[i - start_step + 1][0]\n"
"                    _w = float(os.environ.get('FF_PRESERVE_W','0.3'))\n"
"                    _m = local_var_reg[0].to(latents.dtype) if local_var_reg.dim()==4 else local_var_reg.to(latents.dtype)\n"
"                    latents[0] = latents[0]*(1 - _w*_m) + _cl*(_w*_m)\n"
"                latents_list.append(latents)"
)
assert pres_old in m, "preservation anchor missing"
m=m.replace(pres_old,pres_new,1)

sig_old = "        last_up_block_idx: int = None,\n    ):"
sig_new = '''        last_up_block_idx: int = None,
        ff_feature_ref = None,
        ff_feature_mask = None,
        ff_feature_beta: float = 0.0,
        ff_feature_mode: str = "hf",
        ff_feature_radius: int = 3,
        ff_feature_mid_low: int = 1,
        ff_feature_mid_high: int = 3,
        ff_feature_mid_weight: float = 1.0,
        ff_feature_high_weight: float = 0.65,
        ff_feature_angle: float = 0.0,
        ff_feature_block: int = 1,
    ):'''
a=replace_once(a,sig_old,sig_new,"generalized spectral signature")

_fuse_token="all_intermediate_features.append(sample)"
_positions=[x.start() for x in re.finditer(re.escape(_fuse_token),a)]
assert len(_positions)==1, f"expected one decoder feature anchor, got {len(_positions)}"
_token_pos=_positions[0]
_line_start=a.rfind("\n",0,_token_pos)+1
_indent=a[_line_start:_token_pos]
assert _indent.strip()==""

_body_lines=[
"# Optional masked spectral/full feature residual fusion.",
"# Batch semantics: [uncond_edit, uncond_ref, cond_edit, cond_ref].",
"if ff_feature_ref is not None and i == int(ff_feature_block) and float(ff_feature_beta) > 0:",
"    _ref=ff_feature_ref.to(device=sample.device,dtype=sample.dtype)",
"    if _ref.shape != sample.shape:",
"        raise RuntimeError(f'feature shape mismatch: ref={_ref.shape}, cur={sample.shape}')",
"",
"    if ff_feature_mask is None:",
"        _mask=torch.ones((sample.shape[0],1,*sample.shape[-2:]),device=sample.device,dtype=sample.dtype)",
"    else:",
"        _mask=ff_feature_mask",
"        if _mask.dim()==2: _mask=_mask[None,None]",
"        elif _mask.dim()==3: _mask=_mask[:,None]",
"        _mask=F.interpolate(_mask.float(),size=sample.shape[-2:],mode='bilinear',align_corners=False).to(device=sample.device,dtype=sample.dtype)",
"        if _mask.shape[0] != sample.shape[0]:",
"            _mask=_mask[:1].expand(sample.shape[0],-1,-1,-1)",
"",
"    _gate=torch.zeros((sample.shape[0],1,1,1),device=sample.device,dtype=sample.dtype)",
"    if sample.shape[0]>=4:",
"        _gate[0]=1; _gate[2]=1",
"    else:",
"        _gate[0]=1",
"    _mask=_mask*_gate",
"",
"    def _fft_select(_x,_mode):",
"        _xf=torch.fft.fftshift(torch.fft.fft2(_x.float(),dim=(-2,-1)),dim=(-2,-1))",
"        _h,_w=_x.shape[-2:]",
"        _yy=torch.arange(_h,device=_x.device)[:,None]-(_h//2)",
"        _xx=torch.arange(_w,device=_x.device)[None,:]-(_w//2)",
"        _rho=torch.sqrt(_yy.float()**2+_xx.float()**2)",
"        _r=float(ff_feature_radius)",
"        _lo=float(ff_feature_mid_low); _hi=float(ff_feature_mid_high)",
"        if _mode=='hf':",
"            _weight=(_rho>_r).float()",
"        elif _mode=='mid':",
"            _weight=((_rho>_lo)&(_rho<=_hi)).float()",
"        elif _mode=='midhf':",
"            _mid=((_rho>_lo)&(_rho<=_hi)).float()*float(ff_feature_mid_weight)",
"            _high=(_rho>_hi).float()*float(ff_feature_high_weight)",
"            _weight=_mid+_high",
"        elif _mode=='dirhf':",
"            _base=(_rho>_r).float()",
"            _phi=torch.atan2(_yy.float(),_xx.float()+1e-8)",
"            _theta=torch.as_tensor(float(ff_feature_angle)*3.141592653589793/180.0,device=_x.device)",
"            _dir=(0.35+0.65*torch.abs(torch.cos(_phi-_theta)))",
"            _weight=_base*_dir",
"        else:",
"            raise ValueError(f'unknown spectral mode={_mode}')",
"        _yf=_xf*_weight[None,None]",
"        _y=torch.fft.ifft2(torch.fft.ifftshift(_yf,dim=(-2,-1)),dim=(-2,-1)).real",
"        return _y.to(dtype=_x.dtype)",
"",
"    _mode=str(ff_feature_mode).lower()",
"    if _mode=='full':",
"        _delta=_ref-sample",
"    else:",
"        _delta=_fft_select(_ref,_mode)-_fft_select(sample,_mode)",
"    sample=sample+float(ff_feature_beta)*_mask*_delta",
]
_insert=''.join(_indent+x+'\n' for x in _body_lines)
a=a[:_line_start]+_insert+a[_line_start:]
print(f"✓ generalized feature insertion found; indent={len(_indent)}")

assert "    def forward_sampling(" in m and "    def prox_regularization" in m
pre,rest=m.split("    def forward_sampling(",1)
body,post=rest.split("    def prox_regularization",1)

# Robustly replace only the UNet call inside forward_sampling.
# This avoids depending on the exact blank-line layout around controller.log_mask.
_unet_pat=re.compile(
    r"(?ms)^(?P<indent>[ \t]+)noise_pred[ \t]*=[ \t]*self\.unet\(\s*"
    r"model_inputs\s*,\s*t\s*,\s*encoder_hidden_states\s*=\s*text_embeddings\s*\)\s*$"
)
_um=list(_unet_pat.finditer(body))
if len(_um)!=1:
    _cands=[ln for ln in body.splitlines() if 'noise_pred' in ln and 'self.unet' in ln]
    raise AssertionError(f"forward_sampling UNet call count={len(_um)}; candidates={_cands[:8]}")
_um=_um[0]
noise_new="            _ff_ref=None\n            _ff_beta=0.0\n            if os.environ.get('FF_HFF_ACTIVE','0')=='1':\n                _denom=max(1.0,float((num_inference_steps-1)-start_step))\n                _progress=float(i-start_step)/_denom\n                _horizon=float(os.environ.get('FF_HFF_HORIZON','0.55'))\n                _beta0=float(os.environ.get('FF_HFF_BETA','0.30'))\n                if _horizon>0 and _progress<=_horizon:\n                    _ff_beta=_beta0*0.5*(1.0+np.cos(np.pi*_progress/_horizon))\n\n                if _ff_beta>1e-8:\n                    _ref_mode=os.environ.get('FF_HFF_REF_MODE','same').lower()\n                    if _ref_mode=='fixed':\n                        _span=max(1,(num_inference_steps-1)-start_step)\n                        _frac=float(os.environ.get('FF_HFF_REF_FRAC','0.35'))\n                        _ri=1+int(round(_frac*_span))\n                        _ri=max(1,min(len(refer_latents)-1,_ri))\n                        _si=max(0,min(len(self.scheduler.timesteps)-1,start_step+_ri-1))\n                        _t_ref=self.scheduler.timesteps[_si]\n                    else:\n                        _ri=i-start_step+1\n                        _t_ref=t\n\n                    _coarse_lat=refer_latents[_ri][0:1]\n                    _src_lat=refer_latents[_ri][1:2]\n                    _ref_pair=torch.cat([_coarse_lat,_src_lat],dim=0)\n                    _ref_inputs=torch.cat([_ref_pair]*2,dim=0)\n\n                    _saved_att=getattr(self.controller,'cur_att_layer',None)\n                    _saved_step=getattr(self.controller,'cur_step',None)\n                    _ff_list=self.unet(\n                        _ref_inputs,_t_ref,\n                        encoder_hidden_states=text_embeddings,\n                        last_up_block_idx=int(os.environ.get('FF_HFF_BLOCK','1'))\n                    )\n                    if _saved_att is not None: self.controller.cur_att_layer=_saved_att\n                    if _saved_step is not None: self.controller.cur_step=_saved_step\n                    _ff_ref=_ff_list[-1].detach()\n\n            noise_pred=self.unet(\n                model_inputs,t,\n                encoder_hidden_states=text_embeddings,\n                ff_feature_ref=_ff_ref,\n                ff_feature_mask=getattr(self.controller,'fg_retain_mask_st2',None),\n                ff_feature_beta=_ff_beta,\n                ff_feature_mode=os.environ.get('FF_HFF_MODE','hf'),\n                ff_feature_radius=int(os.environ.get('FF_HFF_RADIUS','3')),\n                ff_feature_mid_low=int(os.environ.get('FF_HFF_MID_LOW','1')),\n                ff_feature_mid_high=int(os.environ.get('FF_HFF_MID_HIGH','3')),\n                ff_feature_mid_weight=float(os.environ.get('FF_HFF_MID_WEIGHT','1.0')),\n                ff_feature_high_weight=float(os.environ.get('FF_HFF_HIGH_WEIGHT','0.65')),\n                ff_feature_angle=float(os.environ.get('FF_CASE_RZ','0.0')),\n                ff_feature_block=int(os.environ.get('FF_HFF_BLOCK','1')),\n            )"
# Re-indent the replacement to the exact indentation of the matched UNet call.
# The original call lives inside `with torch.no_grad():` (normally 16 spaces);
# hard-coding 12 spaces prematurely exits that block and makes the following
# noise_pred_uncon/noise_pred_con line an unexpected indent.
_noise_indent=_um.group("indent")
_noise_lines=noise_new.splitlines()
_noise_non=[ln for ln in _noise_lines if ln.strip()]
_noise_min=min(len(ln)-len(ln.lstrip(" ")) for ln in _noise_non)
noise_new="\n".join(
    (_noise_indent+ln[_noise_min:]) if ln.strip() else ""
    for ln in _noise_lines
)
print(f"✓ replacement re-indented to {len(_noise_indent)} spaces")
body=body[:_um.start()]+noise_new+body[_um.end():]
print("✓ forward_sampling UNet call patched via regex")
m=pre+"    def forward_sampling("+body+"    def prox_regularization"+post

open(S,"w").write(s)
open(M,"w").write(m)
open(A,"w").write(a)

for _f in (S,M,A):
    py_compile.compile(_f,doraise=True)

import torch
_x=torch.randn(1,4,16,16)
_xf=torch.fft.fftshift(torch.fft.fft2(_x.float(),dim=(-2,-1)),dim=(-2,-1))
assert _xf.shape==_x.shape
print("✓ FINAL A patch installed + syntax-compiled")
print("✓ modes: hf / full / mid / midhf / dirhf")
print("✓ reference modes: same / fixed")


✓ generalized feature insertion found; indent=12
✓ replacement re-indented to 16 spaces
✓ forward_sampling UNet call patched via regex
✓ FINAL A patch installed + syntax-compiled
✓ modes: hf / full / mid / midhf / dirhf
✓ reference modes: same / fixed


## 7. Dataset — exact historical balanced-200; manifests for every task

In [7]:

# ===== DATA: exact historical balanced-200 + all task manifests =====
import os, glob, json, csv, random, shutil, hashlib
from collections import defaultdict, Counter

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
           if os.path.isdir(f"{c}/source_img"))
COARSE=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png",recursive=True)[0].split("/coarse_img/")[0]+"/coarse_img"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(
    glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
ANNs=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]
META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if not os.path.exists(d):
        os.symlink(f"{CACHE}/{nm}",d)

for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if not os.path.exists(d):
        os.symlink(sc,d)

shutil.copy(ANNs,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META))
      if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]

random.seed(42)
cells=defaultdict(list)
for r in meta:
    cells[(r["edit_type"],r["difficulty"])].append(r)

keys=sorted(cells)
per=200//len(keys)
picked=[]
for k in keys:
    pool=cells[k][:]
    random.shuffle(pool)
    picked += pool[:per]

chosen={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in chosen]
random.shuffle(left)
for r in left:
    if len(picked)>=200:
        break
    picked.append(r)
picked=picked[:200]

counts=Counter(r["edit_type"] for r in picked)
assert len(picked)==200
assert counts["move"]==67 and counts["resize"]==67 and counts["rotate"]==66, counts
json.dump(picked,open(f"{GEO}/subset_meta.json","w"),indent=2)

fp_rows=sorted(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}" for r in picked)
SUBSET_SHA256=hashlib.sha256("\n".join(fp_rows).encode()).hexdigest()
print("balanced-200:",dict(counts))
print("balanced-200 SHA256:",SUBSET_SHA256)

def build_manifest(rows,path):
    o={}
    for r in rows:
        d,i,e=r["da_n"],r["ins_id"],r["case_id"]
        lf=dict(ann[d]["instances"][i][e])
        lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"])
        lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    json.dump(o,open(path,"w"))
    return path

build_manifest(picked,f"{GEO}/gen_subset.json")
for et in ["move","rotate","resize"]:
    rows=[r for r in picked if r["edit_type"]==et]
    build_manifest(rows,f"{GEO}/gen_{et}.json")
    json.dump(rows,open(f"{GEO}/subset_meta_{et}.json","w"),indent=2)

os.makedirs(f"{GEO}/gen_eval",exist_ok=True)
d=f"{GEO}/gen_eval/baseline"
if os.path.islink(d):
    os.remove(d)
elif os.path.exists(d):
    shutil.rmtree(d)
os.symlink(GENBASE,d)
print("baseline linked:",GENBASE)


balanced-200: {'move': 67, 'resize': 67, 'rotate': 66}
balanced-200 SHA256: 3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c
baseline linked: /kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen/gen_results_2d_final/gen_results_2d_backup


## 8. Baseline validation gate

In [8]:
# ===== C7 validation gate (defaults must reproduce baseline) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5:break
        if n>=5:break
    if n>=5:break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation 0 images")
diffs=[float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{os.path.relpath(n,'/kaggle/temp/val5')}").convert("RGB").resize(Image.open(n).size),float)).mean()) for n in imgs]
print("validation mean|Δ|:",[round(x,3) for x in diffs]); assert max(diffs)<1.0,"NOT REPRODUCING"; print("✓ validation passed")

validation mean|Δ|: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ validation passed


## 9. Exhaustive T4×2 generation — every structural model on all tasks

In [9]:

# ===== FINAL A GENERATION: all models on all 200 tasks, T4x2, resume-safe =====
import os, time, glob, subprocess, socket, json
from collections import deque

GEO="/kaggle/temp/GeoBenchMeta"
P="/kaggle/temp/FreeFine/evaluation/FreeFine"
OUT="/kaggle/working/finalA_structural/variants"
os.makedirs(OUT,exist_ok=True)

GEN_SOFT_DEADLINE=NB_START+9.25*3600
GEN_HARD_STOP=NB_START+9.75*3600

ALL_TYPES="move,rotate,resize"
COMMON={
    "FF_ROUTER":1,
    "FF_HFF_TYPES":ALL_TYPES,
    "FF_HFF_HORIZON":0.55,
    "FF_HFF_BETA":0.30,
    "FF_HFF_RADIUS":3,
    "FF_HFF_BLOCK":1,
}

JOBS=[
    ("R1_ALL",f"{GEO}/gen_subset.json",200,{"FF_ROUTER":1}),
    ("HF30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"hf"}),
    ("FULL30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"full"}),
    ("MID30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"mid",
        "FF_HFF_MID_LOW":1,"FF_HFF_MID_HIGH":3}),
    ("MIDHF30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"midhf",
        "FF_HFF_MID_LOW":1,"FF_HFF_MID_HIGH":3,"FF_HFF_MID_WEIGHT":1.0,"FF_HFF_HIGH_WEIGHT":0.65}),
    ("FIXREF_HF30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"hf",
        "FF_HFF_REF_MODE":"fixed","FF_HFF_REF_FRAC":0.35}),
    ("FIXREF_MIDHF30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"midhf",
        "FF_HFF_REF_MODE":"fixed","FF_HFF_REF_FRAC":0.35,
        "FF_HFF_MID_LOW":1,"FF_HFF_MID_HIGH":3,"FF_HFF_HIGH_WEIGHT":0.65}),
    ("DIRHF30_ALL",f"{GEO}/gen_subset.json",200,{**COMMON,"FF_HFF":1,"FF_HFF_MODE":"dirhf"}),
]

priority={
    "HF30_ALL":0,
    "MIDHF30_ALL":1,
    "FIXREF_HF30_ALL":2,
    "FIXREF_MIDHF30_ALL":3,
    "DIRHF30_ALL":4,
    "FULL30_ALL":5,
    "MID30_ALL":6,
    "R1_ALL":7,
}

json.dump(
    [{"tag":t,"subset":os.path.basename(sub),"expected":n,"env":e} for t,sub,n,e in JOBS],
    open("/kaggle/working/finalA_manifest.json","w"),
    indent=2
)

def count_images(path):
    return len(glob.glob(path+"/**/*.png",recursive=True))

def free_port():
    s=socket.socket()
    s.bind(("",0))
    p=s.getsockname()[1]
    s.close()
    return p

def launch(tag,subset,expected,opts,gpu):
    out=f"{OUT}/{tag}"
    os.makedirs(out,exist_ok=True)
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],
        "PYTHONUNBUFFERED":"1",
        "TOKENIZERS_PARALLELISM":"false",
        "HF_HOME":"/kaggle/temp/hf",
        "NCCL_P2P_DISABLE":"1",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_SUBSET_JSON":subset,
        "FF_OUT_DIR":out,
    })
    env.update({k:str(v) for k,v in opts.items()})
    log=open(f"{out}/log.txt","a")
    proc=subprocess.Popen(
        ["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1",
         "--master-port",str(free_port()),"freefine_sweep_2d.py"],
        cwd=P,env=env,stdout=log,stderr=subprocess.STDOUT
    )
    return {"tag":tag,"expected":expected,"out":out,"gpu":gpu,
            "proc":proc,"log":log,"start":time.time()}

queue=deque(sorted(
    [j for j in JOBS if count_images(f"{OUT}/{j[0]}")<j[2]],
    key=lambda j:priority[j[0]]
))
running={}
done=[]
failed=[]

while queue or running:
    for gpu,info in list(running.items()):
        rc=info["proc"].poll()
        if rc is not None:
            info["log"].close()
            n=count_images(info["out"])
            print(
                f"[GPU{gpu}] {info['tag']} rc={rc} images={n}/{info['expected']} "
                f"time={(time.time()-info['start'])/60:.1f}m",
                flush=True
            )
            if rc==0 and n>=info["expected"]:
                done.append(info["tag"])
            else:
                failed.append(info["tag"])
                try:
                    print(open(f"{info['out']}/log.txt").read()[-3000:],flush=True)
                except Exception:
                    pass
            del running[gpu]

    if time.time()>GEN_HARD_STOP and running:
        print("GEN_HARD_STOP",flush=True)
        for gpu,info in list(running.items()):
            info["proc"].terminate()
        time.sleep(8)
        for gpu,info in list(running.items()):
            if info["proc"].poll() is None:
                info["proc"].kill()
            info["log"].close()
            failed.append(info["tag"])
            del running[gpu]
        break

    for gpu in [0,1]:
        if gpu not in running and queue and time.time()<GEN_SOFT_DEADLINE:
            j=queue.popleft()
            running[gpu]=launch(*j,gpu)
            print(f"[GPU{gpu}] launched {j[0]}",flush=True)

    if queue and not running and time.time()>=GEN_SOFT_DEADLINE:
        break
    time.sleep(15)

status={
    "done":done,
    "failed":failed,
    "not_launched":[j[0] for j in queue],
    "partial":{j[0]:count_images(f"{OUT}/{j[0]}") for j in JOBS},
}
json.dump(status,open("/kaggle/working/finalA_generation_status.json","w"),indent=2)
print(json.dumps(status,indent=2))


[GPU0] launched HF30_ALL
[GPU1] launched MIDHF30_ALL
[GPU1] MIDHF30_ALL rc=0 images=200/200 time=147.5m
[GPU1] launched FIXREF_HF30_ALL
[GPU0] HF30_ALL rc=0 images=200/200 time=147.8m
[GPU0] launched FIXREF_MIDHF30_ALL
[GPU1] FIXREF_HF30_ALL rc=0 images=200/200 time=148.0m
[GPU0] FIXREF_MIDHF30_ALL rc=0 images=200/200 time=147.8m
[GPU0] launched DIRHF30_ALL
[GPU1] launched FULL30_ALL
[GPU0] DIRHF30_ALL rc=0 images=200/200 time=147.8m
[GPU1] FULL30_ALL rc=0 images=200/200 time=147.8m
[GPU0] launched MID30_ALL
[GPU1] launched R1_ALL
[GPU1] R1_ALL rc=0 images=200/200 time=131.0m
GEN_HARD_STOP
{
  "done": [
    "MIDHF30_ALL",
    "HF30_ALL",
    "FIXREF_HF30_ALL",
    "FIXREF_MIDHF30_ALL",
    "DIRHF30_ALL",
    "FULL30_ALL",
    "R1_ALL"
  ],
  "failed": [
    "MID30_ALL"
  ],
  "not_launched": [],
  "partial": {
    "R1_ALL": 200,
    "HF30_ALL": 200,
    "FULL30_ALL": 200,
    "MID30_ALL": 180,
    "MIDHF30_ALL": 200,
    "FIXREF_HF30_ALL": 200,
    "FIXREF_MIDHF30_ALL": 200,
    "DIRHF

## 10. Standardized evaluation — all tasks and all core metrics

In [10]:

# ===== STANDARDIZED ALL-TASK / ALL-METRIC EVALUATION =====
import os, json, re, glob, subprocess, time, numpy as np, shutil, csv
from PIL import Image

GEO="/kaggle/temp/GeoBenchMeta"
MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
OUT="/kaggle/working/finalA_structural/variants"
RESULT_JSON="/kaggle/working/finalA_results.json"
RESULT_CSV="/kaggle/working/finalA_results.csv"
EVAL_DEADLINE=NB_START+11.65*3600

ann=json.load(open(f"{GEO}/annotation_2d.json"))
picked=json.load(open(f"{GEO}/subset_meta.json"))
SETS=['R1_ALL', 'HF30_ALL', 'FULL30_ALL', 'MID30_ALL', 'MIDHF30_ALL', 'FIXREF_HF30_ALL', 'FIXREF_MIDHF30_ALL', 'DIRHF30_ALL']

for tag in SETS:
    p=f"{OUT}/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True):
        d=f"{GEO}/gen_eval/{tag}"
        if os.path.islink(d):
            os.remove(d)
        elif os.path.exists(d):
            shutil.rmtree(d)
        os.symlink(p,d)

def key(r):
    return (r["da_n"],r["ins_id"],r["case_id"])

def mem(pred):
    return [key(r) for r in picked if pred(r)]

groups={
    "move_all":mem(lambda r:r["edit_type"]=="move"),
    "move_hard":mem(lambda r:r["edit_type"]=="move" and r["difficulty"]=="hard"),
    "rotate_all":mem(lambda r:r["edit_type"]=="rotate"),
    "rotate_hard":mem(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),
    "resize_all":mem(lambda r:r["edit_type"]=="resize"),
    "resize_nonhard":mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]!="hard"),
    "resize_hard":mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),
    "all_200":mem(lambda r:True),
}

# All seven metrics are obtained for each complete task and all_200:
# FID, FID_DINO/FDD, FID_KD, SUBC, BGC, WRAP_E, MD.
# Hard/nonhard severity groups keep the primary four because FID-family at n~22 is not meaningful.
fid_groups={"move_all","rotate_all","resize_all","all_200"}

def wrap_e(ids,gd):
    tot=0.0
    n=0
    for d,i,e in ids:
        cp=f"{GEO}/Geo-Bench-2D/coarse_img/{d}/{i}/{e}.png"
        gp=f"{gd}/{d}/{i}/{e}.png"
        tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"
        if not all(os.path.exists(x) for x in (cp,gp,tp)):
            continue
        C=np.array(Image.open(cp).convert("RGB"),float)/255.0
        G=np.array(Image.open(gp).convert("RGB"),float)/255.0
        T=np.array(Image.open(tp).convert("L"),float)/255.0
        if G.shape[:2]!=C.shape[:2]:
            G=np.array(
                Image.fromarray((G*255).astype("uint8")).resize((C.shape[1],C.shape[0])),
                float
            )/255.0
        if T.shape[:2]!=C.shape[:2]:
            T=np.array(
                Image.fromarray((T*255).astype("uint8")).resize((C.shape[1],C.shape[0])),
                float
            )/255.0
        mm=np.repeat(T[...,None],3,axis=2)
        su=mm.sum()
        if su<=0:
            continue
        tot+=float(np.sum(np.abs(C*mm-G*mm))/su)
        n+=1
    return round(tot/n,4) if n else None

def manifest(setn,gname,ids):
    o={}
    base=f"{GEO}/gen_eval/{setn}"
    used=0
    for d,i,e in ids:
        if not os.path.exists(f"{base}/{d}/{i}/{e}.png"):
            continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    p=f"{GEO}/m_{setn}_{gname}.json"
    json.dump(o,open(p,"w"))
    return p,used

def run_metric(manp,task):
    env=os.environ.copy()
    env.update({
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_MD_SEED":"42",
    })
    out=subprocess.run(
        [PY,"main.py","--path",manp,"--use_relative_path","--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2",
         "--task",task,"--level","0"],
        cwd=MET,env=env,capture_output=True,text=True
    )
    txt=out.stdout+out.stderr
    vals={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","MD"]:
        hits=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
        if hits:
            vals[k]=round(float(hits[-1]),4)
    if out.returncode!=0:
        vals["_rc"]=out.returncode
        vals["_tail"]=txt[-1200:]
    return vals

eval_sets=["baseline"]+[s for s in SETS if os.path.exists(f"{GEO}/gen_eval/{s}")]
results={}
for sn in eval_sets:
    if time.time()>EVAL_DEADLINE:
        print("EVAL_DEADLINE",flush=True)
        break
    results[sn]={}
    for g,ids in groups.items():
        if time.time()>EVAL_DEADLINE:
            break
        manp,used=manifest(sn,g,ids)
        if used==0:
            continue

        v=run_metric(manp,"000110100")
        if g in fid_groups and used==len(ids) and time.time()<EVAL_DEADLINE:
            v.update(run_metric(manp,"100110011"))

        v["WRAP_E"]=wrap_e(ids,f"{GEO}/gen_eval/{sn}")
        v["n"]=used
        results[sn][g]=v
        print(f"{sn:26s} {g:14s} -> {v}",flush=True)
        json.dump(results,open(RESULT_JSON,"w"),indent=2)

fields=["set","group","SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD","n"]
rows=[]
for sn,gd in results.items():
    for g,v in gd.items():
        rows.append({"set":sn,"group":g,**{k:v.get(k) for k in fields[2:]}})

with open(RESULT_CSV,"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=fields)
    w.writeheader()
    w.writerows(rows)

print("saved:",RESULT_JSON,RESULT_CSV)


baseline                   move_all       -> {'SUBC': 0.959, 'BGC': 0.9639, 'MD': 3.408, 'FID_DINO': 2444.5409, 'FID_KD': 0.0388, 'FID': 200.0405, 'WRAP_E': 0.0495, 'n': 67}
baseline                   move_hard      -> {'SUBC': 0.9545, 'BGC': 0.9617, 'MD': 3.1192, 'WRAP_E': 0.0407, 'n': 22}
baseline                   rotate_all     -> {'SUBC': 0.8962, 'BGC': 0.9684, 'MD': 9.6294, 'FID_DINO': 2502.4414, 'FID_KD': 0.1137, 'FID': 205.9607, 'WRAP_E': 0.0423, 'n': 66}
baseline                   rotate_hard    -> {'SUBC': 0.8523, 'BGC': 0.9664, 'MD': 14.0096, 'WRAP_E': 0.0416, 'n': 22}
baseline                   resize_all     -> {'SUBC': 0.8906, 'BGC': 0.9647, 'MD': 10.9331, 'FID_DINO': 2465.2808, 'FID_KD': 0.112, 'FID': 201.2628, 'WRAP_E': 0.0544, 'n': 67}
baseline                   resize_nonhard -> {'SUBC': 0.9176, 'BGC': 0.9655, 'MD': 7.5773, 'WRAP_E': 0.0523, 'n': 44}
baseline                   resize_hard    -> {'SUBC': 0.8391, 'BGC': 0.9631, 'MD': 19.4713, 'WRAP_E': 0.0585, 'n': 23}


## 11. Compact result matrix

In [11]:

# ===== FINAL A SUMMARY MATRIX =====
import json, os
P="/kaggle/working/finalA_results.json"
assert os.path.exists(P), "results missing"
R=json.load(open(P))
order=["baseline","R1_ALL","HF30_ALL","FULL30_ALL","MID30_ALL","MIDHF30_ALL",
       "FIXREF_HF30_ALL","FIXREF_MIDHF30_ALL","DIRHF30_ALL"]

for g in ["move_all","rotate_all","rotate_hard","resize_all","resize_hard","all_200"]:
    print("\n===",g,"===")
    print(f"{'set':24s} {'SUBC':>8s} {'BGC':>8s} {'WE':>8s} {'MD':>9s} "
          f"{'FID':>9s} {'FDD':>10s} {'KD':>8s}")
    for s in order:
        if s in R and g in R[s]:
            d=R[s][g]
            print(
                f"{s:24s} {str(d.get('SUBC','-')):>8s} {str(d.get('BGC','-')):>8s} "
                f"{str(d.get('WRAP_E','-')):>8s} {str(d.get('MD','-')):>9s} "
                f"{str(d.get('FID','-')):>9s} {str(d.get('FID_DINO','-')):>10s} "
                f"{str(d.get('FID_KD','-')):>8s}"
            )

print("\nNo automatic winner is selected here.")
print("Merge with Account B before freezing the final transform-aware model.")



=== move_all ===
set                          SUBC      BGC       WE        MD       FID        FDD       KD
baseline                    0.959   0.9639   0.0495     3.408  200.0405  2444.5409   0.0388
R1_ALL                      0.961   0.9624   0.0468    3.5176  201.0319  2441.7825   0.0373

=== rotate_all ===
set                          SUBC      BGC       WE        MD       FID        FDD       KD
baseline                   0.8962   0.9684   0.0423    9.6294  205.9607  2502.4414   0.1137
R1_ALL                     0.8962   0.9684   0.0423    9.6294  205.9607  2502.4414   0.1248

=== rotate_hard ===
set                          SUBC      BGC       WE        MD       FID        FDD       KD
baseline                   0.8523   0.9664   0.0416   14.0096         -          -        -
R1_ALL                     0.8523   0.9664   0.0416   14.0096         -          -        -

=== resize_all ===
set                          SUBC      BGC       WE        MD       FID        FDD       KD
b